# 03 - PD scorecard and challenger models

This notebook walks through the probability-of-default build:

1. WoE binning and Information Value, then variable selection (IV, correlation, VIF);
2. an interpretable logistic scorecard on the WoE scale (coefficients + odds ratios)
   with a probit link comparison, plus L1/L2 regularized logits;
3. tree challengers (decision tree, random forest, LightGBM) with cross-validation;
4. out-of-time validation (AUC/Gini/KS, ROC, KS curve, calibration, confusion matrix
   at a KS-optimal cut-off), population-stability indices and SHAP.

The heavy lifting is run by `python -m src.pipeline pd`, which persists every
table under `reports/results`. Here we load and interpret those artifacts, and
re-run a small WoE demonstration live.

In [ ]:
import json, pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Image, display
RES = ROOT / 'reports' / 'results'
pd.set_option('display.width', 180)
summary = json.loads((RES / 'summary.json').read_text()) if (RES / 'summary.json').exists() else {}
print('summary.json present:', bool(summary))

In [ ]:
def load(name):
    p = RES / name
    return pandas_read(p) if p.exists() else pd.DataFrame()

def pandas_read(p):
    return pd.read_csv(p)

pd_info = summary.get('pd', {})
print('train:', pd_info.get('train_years'), 'n =', pd_info.get('n_train'), ' default rate =', pd_info.get('default_rate_train'))
print('oot:  ', pd_info.get('oot_years'), 'n =', pd_info.get('n_oot'), ' default rate =', pd_info.get('default_rate_oot'))
print('variables binned:', pd_info.get('n_variables_binned'), ' used:', pd_info.get('n_features_used'))

In [ ]:
display(load('pd_iv_summary.csv').head(20))
pd.Series(pd_info.get('selected_variables', []), name='selected variable')

In [ ]:
display(load('pd_model_comparison.csv'))
coef = load('pd_logit_coefficients.csv')
coef['abs_coef'] = coef['coefficient'].abs()
display(coef.sort_values('abs_coef', ascending=False)[['variable', 'coefficient', 'std_err', 'p_value', 'odds_ratio']].round(4))

Odds ratios are read per one-unit change in **WoE**: a coefficient of `b`
means the odds of default multiply by `exp(b)` across adjacent WoE levels.
Because WoE increases with risk, a positive coefficient is expected for every
monotone variable.

In [ ]:
display(load('pd_calibration_champion.csv'))
display(load('pd_psi_champion.csv').head(20))
print('confusion at KS-optimal cut-off:', pd_info.get('confusion_oot'))

In [ ]:
pts = load('pd_scorecard_points.csv')
display(pts.head(20))
display(load('pd_shap_importance.csv').head(15))

In [ ]:
for name in ['roc_curves.png', 'ks_curve.png', 'calibration.png', 'score_distribution.png',
             'scorecard_distribution.png', 'shap_summary.png']:
    p = ROOT / 'reports' / 'figures' / name
    if p.exists():
        display(Image(filename=str(p)))

### Live WoE demonstration

A quick, self-contained re-fit of the binning on a subsample, to show how a
single variable is binned and scored.

In [ ]:
from src.config import Config
from src.pipeline import load_modeling_frame, train_oot_split
from src import features
from src.woe_iv import fit_numeric, fit_categorical, iv_summary

cfg = Config.load()
df = load_modeling_frame(cfg)
train, oot = train_oot_split(df, cfg)
sub = train.sample(n=min(60_000, len(train)), random_state=cfg.seed)
enc = fit_numeric(sub['dti'], sub['default_flag'], 'dti', max_bins=6)
display(enc.table[['bin', 'n', 'bad_rate', 'woe', 'iv']].round(4))
print('IV(dti) =', round(enc.iv, 4))

In [ ]:
rows = []
for col in ['loan_amnt_to_income', 'installment_to_income', 'annual_inc', 'dti', 'revol_util']:
    e = fit_numeric(sub[col], sub['default_flag'], col, max_bins=6)
    rows.append({'variable': col, 'iv': e.iv, 'bins': len(e.table)})
for col in ['term', 'home_ownership', 'purpose', 'addr_state']:
    e = fit_categorical(sub[col], sub['default_flag'], col, max_bins=6)
    rows.append({'variable': col, 'iv': e.iv, 'bins': len(e.table)})
display(pd.DataFrame(rows).sort_values('iv', ascending=False).round(4))

### Limitations

* **Reject inference is out of scope.** The model is estimated on *accepted*
  loans only, so it describes the booked population, not the applicant flow.
* The **primary** out-of-time window is the fully-matured vintages
  (`mature_train_years` / `mature_test_years` in `config.yaml`: train 2010-2013,
  test 2014-2015, 36-month loans only), so the OOT metrics are not biased by
  right-censoring. The late 2017-2018 window is kept as a clearly labeled
  *secondary* analysis in `summary.json -> pd.window_comparison`.
* `grade`/`int_rate` are deliberately excluded from the main scorecard; the
  benchmark AUC in `summary.json` quantifies what the lender's own model adds.